# ImageNet view vs the curated path: why `D_sem_wn` is not built from ResNet predictions

For every pre-SHINE image we build two independent views of "what this thing is" and measure the WordNet shortest-path distance between them:

- **ImageNet view:** the top-3 ResNet-50 predictions, each an ImageNet-1K class and therefore a WordNet synset.
- **Path view:** three concept slots read off the curated path `images/<variant>/<...>/<dirname>/<filename>NN.png`: the filename stem (`tennis3.png` gives `tennis`), the parent directory (`ball`), and the two combined (`tennis ball`).

That gives a 3x3 grid of distances per image. We keep the grid, its minimum, and which cell produced the minimum, because the argmin says whether the model's first guess was usable and whether the filename or the folder carries the concept.

This is a diagnostic, not an RDM builder: it measures the gap between the two views, it does not try to close it. Backend: `imagenet_vs_path.py` (run `python -m analysis.s0_build_rdms.semantic.imagenet_vs_path` once to write `analysis/results/rdms/imagenet_vs_path.csv`; this notebook reads that table).

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[2]  # analysis/s0_build_rdms/semantic/ -> repo root
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from analysis.s0_build_rdms.semantic import imagenet_vs_path as ivp

table = ivp.load_comparison_table()
print(f"{len(table)} images, {int(table['min_distance'].notna().sum())} with at least one scorable cell")

725 images, 717 with at least one scorable cell


## 1. Which cell wins?

For each image, the cell (path slot x ResNet rank) with the smallest WordNet distance.

In [2]:
print(ivp.slot_rank_counts(table).to_string())
print(f"\nmedian minimum distance: {table['min_distance'].median():.1f}")
print(f"exact matches (distance 0): {int((table['min_distance'] == 0).sum())} of {len(table)}")
ivp.plot_slot_rank_heatmap(table).show()

best_rank  1.0  2.0  3.0
best_slot               
filename   338  132   79
dirname     81   58   26
combined     3    0    0

median minimum distance: 3.0
exact matches (distance 0): 117 of 725


## 2. How far is each view?

Minimum distance by the ResNet rank that supplied it, and the marginal distance of each path slot.

In [3]:
print(ivp.slot_marginal_distance(table).round(2).to_string())
ivp.plot_distance_by_rank(table).show()
ivp.plot_distance_by_slot(table).show()

          n_scorable  mean  median  n_exact
slot                                       
filename         682  5.07     4.0      114
dirname          630  5.46     4.0        0
combined          18  6.17     7.0        3


## 3. Agreement with the curated synset, and where it fails

`D_sem_wn` assigns each image its WordNet synset directly (`manifest_synset`). Here we ask how often the path synset that the grid picks (the sense closest to a ResNet prediction) equals that curated synset. The grid chooses whichever sense flatters the model, so this agreement is an upper bound on what automatic path parsing would achieve. ImageNet-1K contains no person, face or human-body class, so the 164 human images are unclassifiable in principle, and the human/not-human split is the clearest illustration of why the ResNet route was dropped.

In [4]:
print("by branch (first two directory levels):")
print(ivp.disagreement_summary(table, by="branch").to_string())
print("\nhuman vs not:")
print(ivp.disagreement_summary(table, by="human").to_string())
ivp.plot_agreement_by_branch(table).show()

by branch (first two directory levels):
                      n  n_scorable  n_agree  n_disagree  agree_pct
group                                                              
inanimate/handmade  267         263      152         111       57.8
animate/human       164         164        6         158        3.7
inanimate/natural   160         157       71          86       45.2
animate/animal      134         133      112          21       84.2

human vs not:
             n  n_scorable  n_agree  n_disagree  agree_pct
group                                                     
non-human  561         553      335         218       60.6
human      164         164        6         158        3.7


In [5]:
detail = ivp.disagreement_detail(table)
print(f"{len(detail)} disagreements")
print(f"pure polysemy (same lemma, different sense): {detail['same_lemma'].mean():.1%}")
print(f"median WordNet distance between the two synsets: {detail['wn_distance'].median():.1f}")
print("\nmost common disagreeing pairs:")
print(detail.groupby(["best_path_synset", "manifest_synset"])
      .agg(n=("curated_path", "size"), wn_distance=("wn_distance", "first"))
      .sort_values("n", ascending=False).head(10).to_string())

376 disagreements
pure polysemy (same lemma, different sense): 37.2%
median WordNet distance between the two synsets: 9.0

most common disagreeing pairs:
                                     n  wn_distance
best_path_synset manifest_synset                   
female.n.01      woman.n.01         49          5.0
male.n.01        man.n.01           37          5.0
hand.n.08        hand.n.01          26         14.0
man.n.10         man.n.01           10         10.0
black.n.07       man.n.01            9          9.0
basketball.n.02  basketball.n.01     8         18.0
black.n.07       woman.n.01          7          9.0
sabra.n.01       prickly_pear.n.01   6         10.0
hammer.n.02      hammer.n.01         6          9.0
banana.n.02      banana.n.01         5         13.0


## Takeaways

1. **The model's top guess supplies the closest match for 59% of scorable images (422 of 717), but the match is rarely exact.** The filename slot supplies the winning cell for 549 of the 717 (the parent directory for 165, the combined slot for 3). The median minimum WordNet distance is 3, and only 117 of 725 images (16%) reach distance 0, almost all through the filename.
2. **The path-derived concept matches the curated synset for 60.6% of non-human images** (animals 84.2%, handmade 57.8%, natural 45.2%), **and for 3.7% of the 164 human images** (6 of 164), where the grid lands on `female`/`male` instead of `woman`/`man`.
3. **Disagreements are mostly not polysemy.** Of 376 disagreements, 37.2% differ only in word sense (same lemma), and the median WordNet distance between the two synsets is 9.
4. **Consequence:** `D_sem_wn` is built from direct label-to-synset assignment (`assign_wordnet_synsets.py`), not from ResNet predictions or path parsing.